In [5]:
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image, ImageDraw, ImageFont

model_id = r"E:\Snapfolia - CS\grounding-dino-tiny"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

image = Image.open("./env_Scramble Egg B (1).jpg")

text = "a leaf. a leaves."

inputs = processor(images=image, text=text, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model(**inputs)

results = processor.post_process_grounded_object_detection(
    outputs,
    inputs.input_ids,
    box_threshold=0.3,
    text_threshold=0.3,
    target_sizes=[image.size[::-1]]
)

# Draw bounding boxes
draw = ImageDraw.Draw(image)
font = ImageFont.load_default()
output_file = "leaf_coordinates.txt"

# Check if any detections were made
if len(results) > 0:
    # Get the first (and likely only) result
    pred_boxes = results[0]["boxes"]
    pred_labels = results[0]["labels"]
    pred_scores = results[0]["scores"]
    
    with open(output_file, 'w') as f:
        # Draw each detected object
        for i, (box, label, score) in enumerate(zip(pred_boxes, pred_labels, pred_scores), 1):
            # Convert box coordinates to integers
            box = [int(b) for b in box]
            
            # Draw the bounding box with increased thickness (width=5)
            draw.rectangle(box, outline="red", width=5)
            
            # Add label and score
            label_text = f"{label}: {score:.2f}"
            draw.text((box[0], box[1]-10), label_text, fill="red", font=font)
            
            # Write coordinates to file
            # Format: class_id x_center y_center width height
            x_center = (box[0] + box[2]) / 2
            y_center = (box[1] + box[3]) / 2
            width = box[2] - box[0]
            height = box[3] - box[1]
            f.write(f"0 {x_center} {y_center} {width} {height}\n")

            # Print detection details
            print(f"Leaf {i}: Score = {score:.2f}, Box = {box}")

        # Print detection details
        print(f"Detected {len(pred_boxes)} objects:")
        for label, score, box in zip(pred_labels, pred_scores, pred_boxes):
            print(f"- {label}: Score = {score:.2f}, Box = {box}")

        # Save or show the image
        image.save("detected_leaves.jpg")
else:
    print("No objects detected.")

Leaf 1: Score = 0.35, Box = [1613, 1563, 1931, 2178]
Leaf 2: Score = 0.39, Box = [71, 949, 3031, 2193]
Leaf 3: Score = 0.33, Box = [1321, 1554, 1581, 2124]
Leaf 4: Score = 0.33, Box = [1330, 959, 1653, 1516]
Leaf 5: Score = 0.34, Box = [1032, 973, 1320, 1507]
Leaf 6: Score = 0.32, Box = [748, 1030, 1010, 1495]
Leaf 7: Score = 0.32, Box = [761, 1528, 1042, 2004]
Leaf 8: Score = 0.32, Box = [1042, 1540, 1325, 2081]
Leaf 9: Score = 0.31, Box = [1895, 984, 2308, 1536]
Leaf 10: Score = 0.31, Box = [1628, 956, 1941, 1521]
Leaf 11: Score = 0.31, Box = [456, 1048, 715, 1481]
Leaf 12: Score = 0.31, Box = [1905, 1576, 2257, 2144]
Leaf 13: Score = 0.30, Box = [2168, 1585, 2600, 2167]
Detected 13 objects:
- a leaf: Score = 0.35, Box = tensor([1613.4033, 1563.2747, 1931.9080, 2178.2915], device='cuda:0')
- a leaves: Score = 0.39, Box = tensor([  71.4980,  949.5917, 3031.3877, 2193.9758], device='cuda:0')
- a leaf: Score = 0.33, Box = tensor([1321.5334, 1554.9766, 1581.7284, 2124.3162], device='cuda